# Thử nghiệm Luật Override — Gate 2 của pipeline

Notebook này thử nghiệm 3 nhánh của **Luật Override** (chạy song song với Random Forest trong Gate 2,
không phải gate riêng — xem lại sơ đồ đã thống nhất):

1. **Tuổi domain** — RDAP (ưu tiên) → WHOIS (fallback)
2. **SSL/TLS** — kiểm tra chứng chỉ có hợp lệ không
3. **Typosquatting** — Levenshtein Distance so với danh sách thương hiệu Việt Nam

Cuối cùng gộp cả 3 thành **luật kết hợp** (đếm cờ vi phạm) đã thiết kế: 0 vi phạm → giữ nguyên theo RF,
1 vi phạm → ép tối thiểu Vùng xám, ≥2 vi phạm → ép thẳng Vùng đen.

**Lưu ý về notebook này:** môi trường tạo notebook bị chặn mạng ra ngoài (`host_not_allowed`), nên
Phần 1 (RDAP/WHOIS) và Phần 2 (SSL) **chưa được tự chạy-xác nhận** bằng kết nối thật — chỉ Phần 3
(Levenshtein, code thuần Python không cần mạng) đã được test kỹ trước khi đưa vào đây. Bạn cần chạy
Phần 1-2 trên máy mình và báo lại nếu lỗi.

## 0. Cài đặt

In [1]:
# Cần cài thêm nếu chưa có (không có sẵn trong môi trường chuẩn):
# pip install python-whois requests

import socket
import ssl
import time
from datetime import datetime, timezone
from urllib.parse import urlparse

import requests

try:
    import whois as pywhois
    HAS_WHOIS = True
except ImportError:
    HAS_WHOIS = False
    print("Chưa cài python-whois — chạy: pip install python-whois")

---
## Phần 1 — Tuổi domain: RDAP trước, WHOIS sau

Đúng thiết kế đã thống nhất: RDAP trả JSON có cấu trúc, ưu tiên dùng trước; chỉ fallback sang WHOIS
(text thô, phải tự parse) khi TLD chưa hỗ trợ RDAP hoặc RDAP lỗi/timeout.

### 1.1. Tra cứu RDAP server đúng cho từng TLD

RDAP không có 1 địa chỉ chung cho mọi domain — mỗi TLD (`.com`, `.vn`, `.org`...) có server RDAP
riêng. IANA duy trì 1 file JSON ánh xạ TLD → server RDAP, cần tra bảng này trước.

In [2]:
IANA_RDAP_BOOTSTRAP = "https://data.iana.org/rdap/dns.json"
_rdap_server_cache = {}

def get_rdap_server(tld, timeout=5):
    """Trả về base URL của RDAP server phụ trách 1 TLD, hoặc None nếu TLD chưa hỗ trợ RDAP."""
    if not _rdap_server_cache:
        resp = requests.get(IANA_RDAP_BOOTSTRAP, timeout=timeout)
        resp.raise_for_status()
        for entry in resp.json().get("services", []):
            tlds, servers = entry[0], entry[1]
            for t in tlds:
                _rdap_server_cache[t.lower()] = servers[0].rstrip("/")
    return _rdap_server_cache.get(tld.lower())

# Test — .com và .vn thường đều đã hỗ trợ RDAP
try:
    print("RDAP server cho .com:", get_rdap_server("com"))
    print("RDAP server cho .vn:", get_rdap_server("vn"))
except Exception as e:
    print(f"Lỗi tra RDAP bootstrap: {e}")

RDAP server cho .com: https://rdap.verisign.com/com/v1
RDAP server cho .vn: None


### 1.2. Gọi RDAP lấy ngày đăng ký domain

In [3]:
def get_domain_age_rdap(domain, timeout=5):
    tld = domain.rsplit(".", 1)[-1]
    server = get_rdap_server(tld)
    if not server:
        return None  # TLD chưa hỗ trợ RDAP, cần fallback WHOIS

    resp = requests.get(f"{server}/domain/{domain}", timeout=timeout)
    if resp.status_code == 404:
        return None  # domain không tồn tại trong RDAP, hoặc registry chưa publish
    resp.raise_for_status()

    data = resp.json()
    for event in data.get("events", []):
        if event.get("eventAction") == "registration":
            reg_date = datetime.fromisoformat(event["eventDate"].replace("Z", "+00:00"))
            age_days = (datetime.now(timezone.utc) - reg_date).days
            return age_days
    return None  # có response nhưng không tìm thấy event registration

try:
    print("Tuổi domain google.com (ngày):", get_domain_age_rdap("google.com"))
except Exception as e:
    print(f"Lỗi RDAP: {e}")

Tuổi domain google.com (ngày): 10580


### 1.3. Fallback WHOIS khi RDAP không có kết quả

In [4]:
def get_domain_age_whois(domain, timeout=5):
    if not HAS_WHOIS:
        return None
    try:
        w = pywhois.whois(domain)
        creation = w.creation_date
        if isinstance(creation, list):  # 1 số domain trả về nhiều ngày, lấy ngày sớm nhất
            creation = min(creation)
        if creation is None:
            return None
        if creation.tzinfo is None:
            creation = creation.replace(tzinfo=timezone.utc)
        return (datetime.now(timezone.utc) - creation).days
    except Exception:
        return None

try:
    print("Tuổi domain google.com qua WHOIS (ngày):", get_domain_age_whois("google.com"))
except Exception as e:
    print(f"Lỗi WHOIS: {e}")

Tuổi domain google.com qua WHOIS (ngày): 10580


### 1.4. Hàm gộp: RDAP trước, WHOIS sau, timeout ngắn

In [5]:
def check_domain_age(domain, timeout=0.5):
    """timeout mặc định 500ms theo đúng lưu ý đã thống nhất trong thiết kế pipeline chính."""
    age_days = None
    try:
        age_days = get_domain_age_rdap(domain, timeout=timeout)
    except Exception:
        pass

    if age_days is None:
        try:
            age_days = get_domain_age_whois(domain, timeout=timeout)
        except Exception:
            pass

    if age_days is None:
        return {"domain": domain, "age_days": None, "violation": None}  # không xác định, không mặc định an toàn

    return {"domain": domain, "age_days": age_days, "violation": age_days < 30}

# Test với vài domain — lưu ý timeout 500ms khá ngắn, real-world nên thử timeout dài hơn khi debug
for d in ["google.com", "vietcombank.com.vn"]:
    print(check_domain_age(d, timeout=5))  # tạm để timeout=5s khi test tay cho chắc, production dùng 0.5s

{'domain': 'google.com', 'age_days': 10580, 'violation': False}
{'domain': 'vietcombank.com.vn', 'age_days': None, 'violation': None}


---
## Phần 2 — SSL/TLS

Lấy chứng chỉ SSL thật của domain qua kết nối TCP trực tiếp cổng 443 — không cần thư viện ngoài,
`ssl` + `socket` đã có sẵn trong Python.

In [6]:
def check_ssl(domain, timeout=5):
    context = ssl.create_default_context()
    try:
        with socket.create_connection((domain, 443), timeout=timeout) as sock:
            with context.wrap_socket(sock, server_hostname=domain) as ssock:
                cert = ssock.getpeercert()

        not_before = datetime.strptime(cert["notBefore"], "%b %d %H:%M:%S %Y %Z").replace(tzinfo=timezone.utc)
        not_after = datetime.strptime(cert["notAfter"], "%b %d %H:%M:%S %Y %Z").replace(tzinfo=timezone.utc)
        issued_days_ago = (datetime.now(timezone.utc) - not_before).days
        is_expired = datetime.now(timezone.utc) > not_after

        return {
            "domain": domain,
            "valid": not is_expired,
            "issued_days_ago": issued_days_ago,
            "issuer": dict(x[0] for x in cert.get("issuer", [])).get("organizationName", "?"),
            # SSL cấp rất gần đây (vài giờ/vài ngày) đi kèm domain mới toanh là dấu hiệu đáng ngờ hơn
            "violation": is_expired or issued_days_ago < 2,
        }
    except (ssl.SSLError, socket.timeout, socket.gaierror, ConnectionRefusedError) as e:
        # Không lấy được chứng chỉ hợp lệ — có thể tự ký, hết hạn nặng, hoặc domain không tồn tại
        return {"domain": domain, "valid": False, "error": str(e), "violation": True}

# Test
for d in ["google.com", "vietcombank.com.vn"]:
    print(check_ssl(d))

{'domain': 'google.com', 'valid': True, 'issued_days_ago': 24, 'issuer': 'Google Trust Services', 'violation': False}
{'domain': 'vietcombank.com.vn', 'valid': True, 'issued_days_ago': 319, 'issuer': 'GlobalSign nv-sa', 'violation': False}


---
## Phần 3 — Typosquatting bằng Levenshtein Distance

Phần này **đã được test kỹ bằng code thuần Python** trước khi đưa vào đây (không cần mạng), dùng đúng
thuật toán quy hoạch động đã giải thích trước đó.

### 3.1. Cài đặt Levenshtein Distance

In [7]:
def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]

# Kiểm chứng lại đúng ví dụ đã giải thích trước đó
assert levenshtein_distance("kitten", "sitting") == 3
print("Test OK — kitten/sitting =", levenshtein_distance("kitten", "sitting"))
print("paypal/paypa1 =", levenshtein_distance("paypal", "paypa1"))

Test OK — kitten/sitting = 3
paypal/paypa1 = 1


### 3.2. Danh sách thương hiệu Việt Nam (khởi đầu — cần tự mở rộng)

Đây chỉ là danh sách mẫu nhỏ để demo — trong pipeline thật cần mở rộng đầy đủ hơn nhiều (ngân hàng,
ví điện tử, sàn TMĐT, viễn thông...), tốt nhất tổng hợp từ NCSC/Chống Lừa Đảo hoặc tự thu thập.

In [8]:
VN_BRAND_DOMAINS = {
    "vietcombank": "vietcombank.com.vn",
    "techcombank": "techcombank.com.vn",
    "momo": "momo.vn",
    "zalopay": "zalopay.vn",
    "shopee": "shopee.vn",
    "tiki": "tiki.vn",
    "lazada": "lazada.vn",
    "fpt": "fpt.vn",
    "viettel": "viettel.vn",
    "mobifone": "mobifone.vn",
}
print(f"Danh sách mẫu: {len(VN_BRAND_DOMAINS)} thương hiệu")

Danh sách mẫu: 10 thương hiệu


### 3.3. Hàm kiểm tra typosquatting

In [9]:
def check_typosquatting(domain, brand_domains=VN_BRAND_DOMAINS, max_distance=2):
    """So domain đang xét với từng domain thương hiệu đã biết.
    Nếu khoảng cách rất nhỏ (<= max_distance) NHƯNG không khớp tuyệt đối -> nghi typosquat."""
    domain = domain.lower().strip()
    closest_brand, closest_domain, min_dist = None, None, float("inf")

    for brand, official_domain in brand_domains.items():
        dist = levenshtein_distance(domain, official_domain)
        if dist < min_dist:
            closest_brand, closest_domain, min_dist = brand, official_domain, dist

    is_exact_match = domain == closest_domain
    is_suspicious = (min_dist <= max_distance) and not is_exact_match

    return {
        "domain": domain,
        "closest_brand": closest_brand,
        "closest_official_domain": closest_domain,
        "distance": min_dist,
        "is_exact_match": is_exact_match,
        "violation": is_suspicious,
    }

# Test 4 trường hợp: domain thật, typosquat rõ, typosquat tinh vi, domain không liên quan
test_domains = [
    "vietcombank.com.vn",       # domain thật -> không vi phạm
    "vietcomb4nk.com.vn",       # thay 'a' bằng '4' -> nghi typosquat
    "momo-vn.com",              # thêm gạch ngang + đổi TLD -> khoảng cách sẽ lớn hơn, thử xem
    "example.com",              # không liên quan gì tới danh sách -> khoảng cách lớn, không vi phạm
]
for d in test_domains:
    print(check_typosquatting(d))

{'domain': 'vietcombank.com.vn', 'closest_brand': 'vietcombank', 'closest_official_domain': 'vietcombank.com.vn', 'distance': 0, 'is_exact_match': True, 'violation': False}
{'domain': 'vietcomb4nk.com.vn', 'closest_brand': 'vietcombank', 'closest_official_domain': 'vietcombank.com.vn', 'distance': 1, 'is_exact_match': False, 'violation': True}
{'domain': 'momo-vn.com', 'closest_brand': 'momo', 'closest_official_domain': 'momo.vn', 'distance': 5, 'is_exact_match': False, 'violation': False}
{'domain': 'example.com', 'closest_brand': 'shopee', 'closest_official_domain': 'shopee.vn', 'distance': 8, 'is_exact_match': False, 'violation': False}


---
## Phần 4 — Gộp cả 3 nhánh: đếm cờ vi phạm

Áp đúng luật kết hợp đã thống nhất: 0 vi phạm → giữ nguyên theo RF, 1 vi phạm → ép tối thiểu Vùng xám,
≥2 vi phạm → ép thẳng Vùng đen. Hàm dưới **tự chạy được**, không cần mạng thật — bạn có thể truyền vào
kết quả RDAP/SSL đã lấy được ở Phần 1-2, hoặc dữ liệu giả lập để test riêng logic kết hợp.

In [10]:
ZONE_THAP, ZONE_XAM, ZONE_DEN = "Vùng thấp", "Vùng xám", "Vùng đen"
ZONE_ORDER = {ZONE_THAP: 0, ZONE_XAM: 1, ZONE_DEN: 2}

def zone_from_rf_score(score):
    if score < 20:
        return ZONE_THAP
    if score < 80:
        return ZONE_XAM
    return ZONE_DEN

def combine_rf_and_override(rf_score, age_result, ssl_result, typo_result):
    zone_from_rf = zone_from_rf_score(rf_score)

    violations = [
        age_result.get("violation"),
        ssl_result.get("violation"),
        typo_result.get("violation"),
    ]
    n_violations = sum(1 for v in violations if v is True)  # None (không xác định) không tính là vi phạm

    if n_violations == 0:
        final_zone = zone_from_rf
    elif n_violations == 1:
        forced_min = ZONE_XAM
        final_zone = forced_min if ZONE_ORDER[forced_min] > ZONE_ORDER[zone_from_rf] else zone_from_rf
    else:  # >= 2
        final_zone = ZONE_DEN

    return {
        "rf_score": rf_score,
        "zone_from_rf_alone": zone_from_rf,
        "n_violations": n_violations,
        "final_zone": final_zone,
    }

### Kiểm chứng lại đúng 3 ví dụ đã bàn trước đó (dữ liệu giả lập, không cần mạng)

In [11]:
# Ví dụ A: RF thấp + 0 vi phạm -> giữ Vùng thấp
result_a = combine_rf_and_override(
    rf_score=15,
    age_result={"violation": False},
    ssl_result={"violation": False},
    typo_result={"violation": False},
)
print("Ví dụ A:", result_a)
assert result_a["final_zone"] == ZONE_THAP

# Ví dụ B: RF thấp (15) NHƯNG domain mới 1 ngày (1 vi phạm) -> bị ép lên Vùng xám
result_b = combine_rf_and_override(
    rf_score=15,
    age_result={"violation": True},   # domain_age = 1 ngày
    ssl_result={"violation": False},
    typo_result={"violation": False},
)
print("Ví dụ B:", result_b)
assert result_b["final_zone"] == ZONE_XAM

# Ví dụ C: RF thấp (20) NHƯNG domain mới + SSL tự ký (2 vi phạm) -> ép thẳng Vùng đen
result_c = combine_rf_and_override(
    rf_score=20,
    age_result={"violation": True},
    ssl_result={"violation": True},
    typo_result={"violation": False},
)
print("Ví dụ C:", result_c)
assert result_c["final_zone"] == ZONE_DEN

print("\nCả 3 ví dụ khớp đúng kết quả đã thống nhất trước đó.")

Ví dụ A: {'rf_score': 15, 'zone_from_rf_alone': 'Vùng thấp', 'n_violations': 0, 'final_zone': 'Vùng thấp'}
Ví dụ B: {'rf_score': 15, 'zone_from_rf_alone': 'Vùng thấp', 'n_violations': 1, 'final_zone': 'Vùng xám'}
Ví dụ C: {'rf_score': 20, 'zone_from_rf_alone': 'Vùng xám', 'n_violations': 2, 'final_zone': 'Vùng đen'}

Cả 3 ví dụ khớp đúng kết quả đã thống nhất trước đó.
